In [16]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import numpy as np
pd.set_option('display.max_columns', None)
pd.options.mode.chained_assignment = None  # default='warn'

# QB Predictions

In [72]:
qb_2023_std_stats = pd.read_csv('data/std_stats/qb_std_stats_2023.csv')
rushing_2023_std_stats = pd.read_csv('data/std_stats/rb_std_stats_2023.csv')
qb_2023_air_yards = pd.read_csv('data/adv_stats/pass_adv_air_yards_2023.csv')
qb_2023_accuracy = pd.read_csv('data/adv_stats/pass_adv_accuracy_2023.csv')
qb_2023_pressure = pd.read_csv('data/adv_stats/pass_adv_pressure_2023.csv')
# qb_2023_playtype = pd.read_csv('data/adv_stats/pass_adv_play_type_2023.csv')  # Not needed

qb_2023 = qb_2023_air_yards.merge(qb_2023_accuracy[['Player', 'Drop%', 'Bad%', 'OnTgt%']], on='Player')
qb_2023 = qb_2023.merge(qb_2023_pressure[['Player','Prss%']], on='Player')
qb_2023 = qb_2023.merge(qb_2023_std_stats[['Player', 'TD%', 'Int%', 'NY/A', 'ANY/A', 'Sk%']], on='Player', how='left')
qb_2023 = qb_2023.merge(rushing_2023_std_stats[['Player', 'Y/G']], on='Player', how='left')
qb_2023 = qb_2023[['Player', 'Drop%', 'Bad%', 'OnTgt%', 'Prss%', 'TD%', 'Int%', 'NY/A', 'ANY/A', 'Sk%', 'Y/G']]

column_mapping = {
    'TD%': 'Passing_TD%',
    'Int%': 'Passing_Int%',
    'NY/A': 'Passing_NY/A',
    'ANY/A': 'Passing_ANY/A', 
    'Sk%': 'Passing_Sk%',
    'Y/G': 'Rushing_Y/G' 
}

# Rename columns
qb_2023.rename(columns=column_mapping, inplace=True)

# Select the relevant columns
qb_2023_x = qb_2023[['Passing_TD%', 'Passing_Int%', 'Passing_NY/A', 'Passing_ANY/A', 'Passing_Sk%', 'Rushing_Y/G', 'Drop%', 'Bad%', 'OnTgt%', 'Prss%']]
qb_2023_x['Rushing_Y/G'] = qb_2023_x['Rushing_Y/G'] / 8.5
qb_2023_x['Passing_Sk%'] = qb_2023_x['Passing_Sk%'] * 2.3

# Convert 'Prss%' to a numerical value by removing the '%' sign and converting to float
qb_2023_x['Prss%'] = qb_2023_x['Prss%'].str.rstrip('%').astype('float') / 100.0

# Convert 'Drop%', 'Bad%', and 'OnTgt%' to numerical values similarly
qb_2023_x['Drop%'] = qb_2023_x['Drop%'].str.rstrip('%').astype('float') / 100.0
qb_2023_x['Bad%'] = qb_2023_x['Bad%'].str.rstrip('%').astype('float') / 100.0
qb_2023_x['OnTgt%'] = qb_2023_x['OnTgt%'].str.rstrip('%').astype('float') / 100.0

model = joblib.load('models/2024/qb_model.pkl')

qb_2023_preds = model.predict(qb_2023_x)

qb_2023['Projections'] = qb_2023_preds

qb_2023_preds_only = qb_2023.sort_values('Projections', ascending=False).reset_index()
qb_2023_preds_only = qb_2023_preds_only[['Player', 'Projections']]

qb_2023_preds_only.to_csv('projections/2024/qb_projections.csv')

/var/folders/22/b81b4c7d6j7gf__0295hzt480000gn/T/ipykernel_89867/594324950.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  qb_2023_x['Rushing_Y/G'] = qb_2023_x['Rushing_Y/G'] / 8.5
/var/folders/22/b81b4c7d6j7gf__0295hzt480000gn/T/ipykernel_89867/594324950.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  qb_2023_x['Passing_Sk%'] = qb_2023_x['Passing_Sk%'] * 2.3
/var/folders/22/b81b4c7d6j7gf__0295hzt480000gn/T/ipykernel_89867/594324950.py:32: SettingWithCopyWarning: 
A value is trying to be set on a 

### Referencing 2022 Models
- QB:
    - Int% == Less than 3%
    - Lng == Greater than 50
    - Y/A == Greater than 7
    - AY/A == Greater than 7
    - Y/C == Greater than 11
    - Sk% == Less than 7
    - NY/A == Greater than 6.5
    - ANY/A == Greater than 5.5
    - Vertical == Greater than 31
    - 3Cone = Less than 7.2
    

In [71]:
# List of RBs with that fit optimal Fantasy RB Criteria
passing_data = pd.read_csv('data/std_stats/qb_std_stats_2023.csv')
qbs = passing_data[passing_data['Pos'] == 'QB']
qbs = qbs[qbs['GS'] >= 4]
qb_combine_stats = pd.read_csv('data/combine_data/pfr_combine_data_2000_thru_2023.csv')

qbs = qbs.merge(qb_combine_stats[['Player', 'Vertical', '3Cone']], on='Player')
qbs = qbs[['Player', 'Int%', 'Lng', 'Y/A', 'AY/A', 'Y/C', 'Sk%', 'NY/A', 'ANY/A', 'Vertical', '3Cone']]

column_mapping = {
    'Int%': 'Passing_Int%',
    'Lng': 'Passing_Lng',
    'Y/A': 'Passing_Y/A',
    'AY/A': 'Passing_AY/A',
    'Y/C': 'Passing_Y/C',
    'Sk%': 'Passing_Sk%',
    'NY/A': 'Passing_NY/A',
    'ANY/A': 'Passing_ANY/A',
}

# Rename columns
qbs.rename(columns=column_mapping, inplace=True)

# Convert 'Prss%' to a numerical value by removing the '%' sign and converting to float
qbs['Passing_Int%'] = qbs['Passing_Int%'] / 100.0
qbs['Passing_Sk%'] = qbs['Passing_Sk%'] / 100.0


# List of QBs with that fit optimal Fantasy QB Criteria
qbs_best_projected = qbs[
                        (qbs['Passing_Int%'] <= 0.03) 
                         & (qbs['Passing_Lng'] >= 50) 
                         & (qbs['Passing_Y/A'] >= 7) 
                         & (qbs['Passing_AY/A'] >= 7) 
                         & (qbs['Passing_Y/C'] >= 11) 
                         & (qbs['Passing_Sk%'] <= 0.07) 
                         & (qbs['Passing_NY/A'] >= 6.5) 
                         & (qbs['Passing_ANY/A'] >= 5.5) 
                         #& (qbs['Vertical'] >= 31) 
                         #& (qbs['3Cone'] <= 7.2)
                         ]


qbs_best_projected = qbs_best_projected[['Player', 'Passing_Int%', 'Passing_Lng', 'Passing_Y/A', 'Passing_AY/A', 'Passing_Y/C', 'Passing_Sk%', 'Passing_NY/A', 'Passing_ANY/A', 'Vertical', '3Cone']]

qbs_best_projected


,Player,Passing_Int%,Passing_Lng,Passing_Y/A,Passing_AY/A,Passing_Y/C,Passing_Sk%,Passing_NY/A,Passing_ANY/A,Vertical,3Cone
0,Jared Goff,0.02,70,7.6,7.7,11.2,0.047,6.89,6.99,27.0,7.17


### QB Preferences
Short List:
- Josh Allen (Clear Number 1)
- Jared Goff 
- Lamar Jackson
- Jordan Love

Flyers:
- Brock Purdy
- Kirk Cousins
- Derek Carr


# RB Predictions

In [43]:
rb_2023_std_stats = pd.read_csv('data/std_stats/rb_std_stats_2023.csv')
rb_2023_std_stats = rb_2023_std_stats[rb_2023_std_stats['Pos'] == 'RB']
rb_2023_std_stats['Att/G'] = rb_2023_std_stats['Att'] / rb_2023_std_stats['G']
rb_2023_actually_played = rb_2023_std_stats[rb_2023_std_stats['Att/G'] >= 5.5]
#rb_2023_adv_stats = pd.read_csv('data/adv_stats/rush_adv_2023.csv')
rec_2023_std_stats = pd.read_csv('data/std_stats/wr_std_stats_2023.csv')
rb_combine_stats = pd.read_csv('data/combine_data/pfr_combine_data_2000_thru_2023.csv')

# qb_2023_playtype = pd.read_csv('data/adv_stats/pass_adv_play_type_2023.csv')  # Not needed

'Rushing_Y/A_y', 'Receiving_Y/Tgt', 'Vertical', 'Age', 'Receiving_Ctch%', '40yd', 'Bench'

rb_2023 = rb_2023_actually_played.merge(rec_2023_std_stats[['Player', 'Ctch%', 'Y/Tgt']], on='Player')
rb_2023 = rb_2023.merge(rb_combine_stats[['Player', 'Vertical', '40yd', 'Bench']], on='Player')
rb_2023 = rb_2023[['Player', 'Age', 'Y/A', 'Ctch%', 'Y/Tgt', 'Vertical', '40yd', 'Bench']]

column_mapping = {
    'Y/A': 'Rushing_Y/A_y',
    'Y/Tgt': 'Receiving_Y/Tgt',
    'Ctch%': 'Receiving_Ctch%'
}

# Rename columns
rb_2023.rename(columns=column_mapping, inplace=True)

# Select the relevant columns
rb_2023_x = rb_2023[['Age', 'Rushing_Y/A_y', 'Receiving_Ctch%', 'Receiving_Y/Tgt', 'Vertical', '40yd', 'Bench']]
#rb_2023['Passing_Sk%'] = rb_2023['Passing_Sk%'] * 2.3

# Convert 'Prss%' to a numerical value by removing the '%' sign and converting to float
rb_2023_x['Receiving_Ctch%'] = rb_2023_x['Receiving_Ctch%'].str.rstrip('%').astype('float') / 100.0

model = joblib.load('models/2024/rb_model.pkl')

rb_2023_preds = model.predict(rb_2023_x)

rb_2023['Projections'] = rb_2023_preds

rb_2023 = rb_2023[rb_2023['Age'] <= 27] # Because they will be 28 in 2024
rb_2023_preds_only = rb_2023.sort_values('Projections', ascending=False).reset_index()
rb_2023_preds_only = rb_2023_preds_only[['Player', 'Projections']]

rb_2023_preds_only.to_csv('projections/2024/rb_projections.csv')

rb_2023_preds_only.head(20)

/var/folders/22/b81b4c7d6j7gf__0295hzt480000gn/T/ipykernel_89867/131737924.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rb_2023_x['Receiving_Ctch%'] = rb_2023_x['Receiving_Ctch%'].str.rstrip('%').astype('float') / 100.0
/opt/homebrew/Caskroom/miniconda/base/envs/sports/lib/python3.8/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but GradientBoostingRegressor was fitted without feature names
  warnings.warn(


,Player,Projections
0,Zach Charbonnet,26.427357
1,Jaylen Warren,26.427357
2,Keaton Mitchell,26.427357
3,Isiah Pacheco,26.208369
4,Rachaad White,26.208369
5,Breece Hall,24.706631
6,Tyler Allgeier,24.706631
7,Zamir White,24.706631
8,Javonte Williams,21.947291
9,Chuba Hubbard,21.947291


### Referencing 2022 models
- RB:
    - Age = 28 or younger
    - 1D = The higher the better, but at least 40
    - Rushing Lng = At least 25
    - Rushing Y/A = At least 4.0
    - Receiving Ctch% = Greater than 75
    - Receiving Lng = Greater than 15
    - Receiving Y/Tgt = Greater than 5
    - Ht = 5'10 or more
    - 40yd = 4.6 or less
    - 3Cone = 7.0 or less
    - Shuttle = 4.25 or less

In [10]:
# List of RBs with that fit optimal Fantasy RB Criteria
rushing_data = pd.read_csv('data/std_stats/rb_std_stats_2023.csv')
rbs = rushing_data[rushing_data['Pos'] == 'RB']
#rb_2023_adv_stats = pd.read_csv('data/adv_stats/rush_adv_2023.csv')
rec_2023_std_stats = pd.read_csv('data/std_stats/wr_std_stats_2023.csv')
rb_combine_stats = pd.read_csv('data/combine_data/pfr_combine_data_2000_thru_2023.csv')

rbs = rbs.merge(rec_2023_std_stats[['Player', 'Ctch%', 'Y/Tgt', 'Lng']], on='Player')
rbs = rbs.merge(rb_combine_stats[['Player', '40yd', 'Ht', 'Shuttle']], on='Player')
rbs = rbs[['Player', 'Age', 'G', '1D', 'Lng_x', 'Lng_y', 'Y/A', 'Ctch%', 'Y/Tgt', 'Shuttle', '40yd', 'Ht']]

column_mapping = {
    '1D': 'Rushing_1D',
    'Lng_x': 'Rushing_Lng',
    'Lng_y': 'Receiving_Lng',
    'Y/A': 'Rushing_Y/A_y',
    'Y/Tgt': 'Receiving_Y/Tgt',
    'Ctch%': 'Receiving_Ctch%'
}

# Rename columns
rbs.rename(columns=column_mapping, inplace=True)

# Convert 'Prss%' to a numerical value by removing the '%' sign and converting to float
rbs['Receiving_Ctch%'] = rbs['Receiving_Ctch%'].str.rstrip('%').astype('float') / 100.0
rbs['Rushing_1D/G'] = rbs['Rushing_1D'] / rbs['G']

rbs['Age_in_2024'] = rbs['Age'] + 1
rbs_best_projected = rbs[(rbs['Age_in_2024'] <= 28) 
                        #& (rbs['Rushing_1D'] >= 40) 
                        & (rbs['Rushing_1D/G'] >= (17/40))
                        & (rbs['Rushing_Lng'] >= 25) 
                        & (rbs['Rushing_Y/A_y'] >= 4) 
                        & (rbs['Receiving_Ctch%'] >= .75) 
                        & (rbs['Receiving_Lng'] >= 15) 
                        & (rbs['Receiving_Y/Tgt'] >= 5) 
                        #& (rbs['Ht'] > 5.8) 
                        #& (rbs['40yd'] < 4.6) 
                        #& (rbs['Shuttle'] < 4.25)
                        ]

rbs_best_projected = rbs_best_projected[['Player', 'Age_in_2024', 'G', 'Rushing_1D/G', 'Rushing_Lng', 'Receiving_Lng', 'Rushing_Y/A_y', 'Receiving_Ctch%', 'Receiving_Y/Tgt', 'Shuttle', '40yd', 'Ht']]

rbs_best_projected

,Player,Age_in_2024,G,Rushing_1D/G,Rushing_Lng,Receiving_Lng,Rushing_Y/A_y,Receiving_Ctch%,Receiving_Y/Tgt,Shuttle,40yd,Ht
6,Breece Hall,23,17,2.352941,83,50,4.5,0.800,6.2,0.00,4.39,5.916667
12,Isiah Pacheco,25,14,3.785714,48,33,4.6,0.898,5.0,0.00,4.37,5.833333
20,Jonathan Taylor,25,10,4.200000,49,40,4.4,0.826,6.7,4.24,4.39,5.833333
22,Jaylen Warren,26,17,2.117647,74,30,5.3,0.824,5.0,0.00,4.55,5.666667
32,Zamir White,25,17,1.117647,43,15,4.3,0.789,5.2,0.00,4.40,6.000000
37,Roschon Johnson,23,15,1.066667,29,24,4.3,0.850,5.2,0.00,4.58,6.083333
48,Keaton Mitchell,22,8,2.375000,60,32,8.4,0.818,8.5,0.00,4.37,5.666667
50,Chase Brown,24,12,0.500000,31,54,4.1,0.933,10.4,0.00,4.43,5.833333


## RB Preferences

### Short List
- Breece Hall
- Isiah Pacheco
- Johnathon Taylor

### Flyers
- Tony Pollard
- Zamir White
- Zach Charbonnet
- Jaylen Warren
- Keaton Mitchell
- Rachaad White
- Tyler Allegier
- Roschon Johnson
- Chase Brown

# WR Projections

In [143]:
rec_2023_std_stats = pd.read_csv('data/std_stats/wr_std_stats_2023.csv')
wr_2023_std_stats = rec_2023_std_stats[rec_2023_std_stats['Pos'] == 'WR']
wr_2023_std_stats['Tgt/G'] = wr_2023_std_stats['Tgt'] / wr_2023_std_stats['G']
wr_2023_actually_played = wr_2023_std_stats[wr_2023_std_stats['Tgt/G'] >= 3]
wr_combine_stats = pd.read_csv('data/combine_data/pfr_combine_data_2000_thru_2023.csv')

wr_2023 = wr_2023_actually_played.merge(rb_combine_stats[['Player', 'Vertical', '40yd', 'Bench', 'Ht', 'Wt', '3Cone', 'Shuttle']], on='Player', how='left')
wr_2023 = wr_2023.fillna(0)
#wr_2023 = wr_2023[['Player', 'Age', 'Ctch%', 'Lng', 'Y/Tgt', '40yd', 'Ht', 'Wt', 'Vertical', 'Bench',  '3Cone', 'Shuttle']]

column_mapping = {
    'Lng': 'Receiving_Lng',
    'Y/Tgt': 'Receiving_Y/Tgt',
    'Ctch%': 'Receiving_Ctch%'
}

# Rename columns
wr_2023.rename(columns=column_mapping, inplace=True)

# Select the relevant columns
wr_2023_x = wr_2023[['Age', 'Receiving_Ctch%', 'Receiving_Lng', 'Receiving_Y/Tgt', '40yd',
       'Ht', 'Wt', 'Vertical', 'Bench', '3Cone', 'Shuttle']]
wr_2023_x['Receiving_Y/Tgt'] = wr_2023_x['Receiving_Y/Tgt'] / 1.2
wr_2023_x['Receiving_Lng'] = wr_2023_x['Receiving_Lng'] / 2.5
wr_2023_x[['40yd',
       'Ht', 'Wt', 'Vertical', 'Bench', '3Cone', 'Shuttle']] = 0

# Convert 'Prss%' to a numerical value by removing the '%' sign and converting to float
wr_2023_x['Receiving_Ctch%'] = wr_2023_x['Receiving_Ctch%'].str.rstrip('%').astype('float') / 100.0

model = joblib.load('models/2024/wr_model.pkl')

wr_2023_preds = model.predict(wr_2023_x)

wr_2023['Projections'] = wr_2023_preds

wr_2023_preds_only = wr_2023.sort_values('Projections', ascending=False).reset_index()
wr_2023_preds_only = wr_2023_preds_only[['Player', 'Projections']]

wr_2023_preds_only.to_csv('projections/2024/wr_projections.csv')

proj_wrs = wr_2023_preds_only['Player'].unique()[:30]

### Referencing 2022 Models
- WR:
    - Age = Less than 30
    - Ctch% = Greater than 50%
    - Lng = Greater than 50 yds
    - Y/Tgt = Greater than 6.5
    - Ht = 5'10 or higher
    - Wt = 200lbs or higher
    - 40yd = 4.6 or less
    - Shutte = 4.3 or less

In [163]:
rec_2023_std_stats = pd.read_csv('data/std_stats/wr_std_stats_2023.csv')
wr_2023_std_stats = rec_2023_std_stats[rec_2023_std_stats['Pos'] == 'WR']
wr_2023_std_stats['Tgt/G'] = wr_2023_std_stats['Tgt'] / wr_2023_std_stats['G']
wr_2023_actually_played = wr_2023_std_stats[wr_2023_std_stats['Tgt/G'] >= 3]
wr_combine_stats = pd.read_csv('data/combine_data/pfr_combine_data_2000_thru_2023.csv')

wr_2023 = wr_2023_actually_played.merge(wr_combine_stats[['Player', 'Vertical', '40yd', 'Bench', 'Ht', 'Wt', '3Cone', 'Shuttle']], on='Player', how='left')
wr_2023 = wr_2023.fillna(0)
wrs = wr_2023[['Player', 'Age', 'Ctch%', 'Lng', 'Y/Tgt', '40yd', 'Ht', 'Wt', 'Vertical', 'Bench',  '3Cone', 'Shuttle']]

column_mapping = {
    'Lng': 'Receiving_Lng',
    'Y/Tgt': 'Receiving_Y/Tgt',
    'Ctch%': 'Receiving_Ctch%'
}

# Rename columns
wrs.rename(columns=column_mapping, inplace=True)

# Convert 'Prss%' to a numerical value by removing the '%' sign and converting to float
wrs['Receiving_Ctch%'] = wrs['Receiving_Ctch%'].str.rstrip('%').astype('float') / 100.0

wrs['Age_in_2024'] = wrs['Age'] + 1

wrs_best_projected = wrs[(wrs['Age_in_2024'] <= 30) 
                         & (wrs['Receiving_Ctch%'] >= 0.5) 
                         & (wrs['Receiving_Lng'] >= 50) 
                         & (wrs['Receiving_Y/Tgt'] >= 6.5) 
                         #& (wrs['Ht'] > 5.8) 
                         #& (wrs['Wt'] > 200) 
                         #& (wrs['40yd'] < 4.6) 
                         #& (wrs['Shuttle'] < 4.3)
                         ]

wrs_best_projected = wrs_best_projected[['Player', 'Age_in_2024', 'Receiving_Ctch%', 'Receiving_Lng', 'Receiving_Y/Tgt', 'Shuttle', '40yd', 'Ht', 'Wt']]
wrs_best_projected['Rk_criteria'] = wrs_best_projected.index + 1

wr_2023_projs = pd.read_csv('projections/2024/wr_projections.csv')
wr_2023_projs['Rk_proj'] = wr_2023_projs.index + 1
#top_proj_wrs = wr_2023_preds_only['Player'].unique()[:30]

#wrs_best_proj_and_critiera = wrs_best_projected[wrs_best_projected['Player'].isin(top_proj_wrs)]
wrs_best_proj_and_critiera = wrs_best_projected.merge(wr_2023_projs[['Player', 'Rk_proj', 'Projections']], on='Player', how='inner')
wrs_best_proj_and_critiera['Combined_Rk'] = (wrs_best_proj_and_critiera['Rk_criteria'] * 0.02) - ((wrs_best_proj_and_critiera['Projections']))
wrs_best_proj_and_critiera = wrs_best_proj_and_critiera.sort_values('Combined_Rk')
wrs_best_proj_and_critiera_with_jjettas = wrs_best_proj_and_critiera[wrs_best_proj_and_critiera['Rk_criteria'] <= 33]
print(wrs_best_proj_and_critiera_with_jjettas['Player'].unique()[:50])
print(wrs_best_proj_and_critiera['Player'].unique()[:50])

['CeeDee Lamb*+' 'Justin Jefferson' 'Tyreek Hill*+' 'Nico Collins'
 'Puka Nacua*' 'Michael Pittman Jr.' "Ja'Marr Chase*"
 'Amon-Ra St. Brown*+' 'D.J. Moore' 'Brandon Aiyuk' 'Chris Olave'
 'Zay Flowers' 'Josh Downs' 'DeVonta Smith' 'Jordan Addison'
 'Jaylen Waddle' 'A.J. Brown*' 'Amari Cooper*' 'Rashee Rice'
 'Calvin Ridley']
['CeeDee Lamb*+' 'Justin Jefferson' 'Tyreek Hill*+' 'Nico Collins'
 'Puka Nacua*' 'Michael Pittman Jr.' "Ja'Marr Chase*"
 'Amon-Ra St. Brown*+' 'D.J. Moore' 'Brandon Aiyuk' 'Chris Olave'
 'Zay Flowers' 'George Pickens' 'Josh Downs' 'DeVonta Smith'
 'Jordan Addison' 'Jaylen Waddle' 'Deebo Samuel' 'Jayden Reed'
 'A.J. Brown*' 'Tyler Boyd' 'Rashid Shaheed*+' 'D.K. Metcalf*'
 'Darius Slayton' 'Tank Dell' 'Amari Cooper*' 'Christian Kirk'
 'Michael Wilson' 'Josh Palmer' 'Rashee Rice' 'Noah Brown' 'Tee Higgins'
 'Diontae Johnson' 'Gabriel Davis' 'Christian Watson' 'Jameson Williams'
 'Calvin Ridley']


### WR Preferences
Short List:
- CeeDee Lamb
- Justin Jefferson 
- Tyreek Hill
- Nico Collins 
- Puka Nacua
- Michael Pittman Jr
- Ja'Marr Chase 
- Amon-Ra St. Brown

2nd Tier:
- D.J. Moore
- Brandon Aiyuk
- Chris Olave
- Zay Flowers 
- George Pickens
- Josh Downs
- DeVonta Smith
- Jordan Addison
- Jaylen Waddle
- A.J. Brown
- Amari Cooper
- Jayden Reed

Flyers:
- Rashee Rice
- Calvin Ridley
- Deebo Samuel
- Tyler Boyd 
- Rashid Shaheed
- D.K. Metcalf
- Darius Slayton
- Tank Dell
- Amari Cooper
- Christian Kirk
- Michael Wilson
- Josh Palmer
- Rashee Rice
- Noah Brown
- Tee Higgins
- Diontae Johnson
- Gabriel Davis
- Christian Watson
- Jameson Williams

# TEs

In [164]:
rec_2023_std_stats = pd.read_csv('data/std_stats/wr_std_stats_2023.csv')
te_2023_std_stats = rec_2023_std_stats[rec_2023_std_stats['Pos'] == 'TE']
te_2023_std_stats['Tgt/G'] = te_2023_std_stats['Tgt'] / te_2023_std_stats['G']
te_2023_actually_played = te_2023_std_stats[te_2023_std_stats['Tgt/G'] >= 1.5]
te_combine_stats = pd.read_csv('data/combine_data/pfr_combine_data_2000_thru_2023.csv')

te_2023 = te_2023_actually_played.merge(te_combine_stats[['Player', 'Vertical', '40yd', 'Bench', 'Ht', 'Wt', '3Cone', 'Shuttle']], on='Player', how='left')
te_2023 = te_2023.fillna(0)
#wr_2023 = wr_2023[['Player', 'Age', 'Ctch%', 'Lng', 'Y/Tgt', '40yd', 'Ht', 'Wt', 'Vertical', 'Bench',  '3Cone', 'Shuttle']]

column_mapping = {
    'Lng': 'Receiving_Lng',
    'Y/Tgt': 'Receiving_Y/Tgt',
    'Ctch%': 'Receiving_Ctch%'
}

# Rename columns
te_2023.rename(columns=column_mapping, inplace=True)

# Select the relevant columns
te_2023_x = te_2023[['Age', 'Receiving_Ctch%', 'Receiving_Lng', 'Receiving_Y/Tgt', '40yd',
       'Ht', 'Wt', 'Vertical', 'Bench', '3Cone', 'Shuttle']]
#te_2023_x['Receiving_Y/Tgt'] = te_2023_x['Receiving_Y/Tgt'] / 1.2
#te_2023_x['Receiving_Lng'] = te_2023_x['Receiving_Lng'] / 2.5
te_2023_x[['40yd',
       'Ht', 'Wt', 'Vertical', 'Bench', '3Cone', 'Shuttle']] = 0

# Convert 'Prss%' to a numerical value by removing the '%' sign and converting to float
te_2023_x['Receiving_Ctch%'] = te_2023_x['Receiving_Ctch%'].str.rstrip('%').astype('float') / 100.0

model = joblib.load('models/2024/wr_model.pkl')

te_2023_preds = model.predict(te_2023_x)

te_2023['Projections'] = te_2023_preds

te_2023_preds_only = te_2023.sort_values('Projections', ascending=False).reset_index()
te_2023_preds_only = te_2023_preds_only[['Player', 'Projections']]

te_2023_preds_only.to_csv('projections/2024/te_projections.csv')

proj_tes = te_2023_preds_only['Player'].unique()[:30]

### Referencing 2022 Models
- TE:
    - Age = Any less than 35
    - Cth% = Greater than 60%
    - Lng = Greater than 30 yds
    - Height = 6'5
    - Weight = 240 to 260
    - 40 = 4.6 to 4.8
    - Vert = 30 to 36
    - 3Cone = Less than 7.2
    - Shuttle = Less than 4.5

In [197]:
rec_2023_std_stats = pd.read_csv('data/std_stats/wr_std_stats_2023.csv')
te_2023_std_stats = rec_2023_std_stats[rec_2023_std_stats['Pos'] == 'TE']
te_2023_std_stats['Tgt/G'] = te_2023_std_stats['Tgt'] / te_2023_std_stats['G']
te_2023_actually_played = te_2023_std_stats[te_2023_std_stats['Tgt/G'] >= 1.5]
te_combine_stats = pd.read_csv('data/combine_data/pfr_combine_data_2000_thru_2023.csv')

te_2023 = te_2023_actually_played.merge(te_combine_stats[['Player', 'Vertical', '40yd', 'Ht', 'Wt', '3Cone', 'Shuttle']], on='Player', how='left')
te_2023 = te_2023.fillna(0)
#wr_2023 = wr_2023[['Player', 'Age', 'Ctch%', 'Lng', 'Y/Tgt', '40yd', 'Ht', 'Wt', 'Vertical', 'Bench',  '3Cone', 'Shuttle']]

column_mapping = {
    'Lng': 'Receiving_Lng',
    'Y/Tgt': 'Receiving_Y/Tgt',
    'Ctch%': 'Receiving_Ctch%'
}

# Rename columns
te_2023.rename(columns=column_mapping, inplace=True)

# Convert 'Prss%' to a numerical value by removing the '%' sign and converting to float
te_2023['Receiving_Ctch%'] = te_2023['Receiving_Ctch%'].str.rstrip('%').astype('float') / 100.0

te_2023['Age_in_2024'] = te_2023['Age'] + 1

tes_best_projected = te_2023[(te_2023['Age_in_2024'] <= 34) 
                         & (te_2023['Receiving_Ctch%'] >= 0.6) 
                         & (te_2023['Receiving_Lng'] >= 30) 
                         & (te_2023['Receiving_Y/Tgt'] >= 6.5) 
                         & (te_2023['Ht'] > 6.3) 
                         & (te_2023['Wt'] > 240) 
                         & (te_2023['40yd'] < 4.8) 
                         & (te_2023['Shuttle'] < 4.5)
                         & (te_2023['Vertical'] > 30) 
                         & (te_2023['3Cone'] < 7.2)
                         ]

tes_best_projected = tes_best_projected[['Player', 'Age_in_2024', 'Receiving_Ctch%', 'Receiving_Lng', 'Receiving_Y/Tgt', 'Shuttle', '40yd', 'Ht', 'Wt', 'Vertical', '3Cone']]
tes_best_projected['Rk_criteria'] = tes_best_projected.index + 1

te_2023_projs = pd.read_csv('projections/2024/te_projections.csv')
te_2023_projs['Rk_proj'] = te_2023_projs.index + 1
#top_proj_tes = te_2023_preds_only['Player'].unique()[:30]

#tes_best_proj_and_critiera = tes_best_projected[tes_best_projected['Player'].isin(top_proj_tes)]
tes_best_proj_and_critiera = tes_best_projected.merge(te_2023_projs[['Player', 'Rk_proj', 'Projections']], on='Player', how='inner')
tes_best_proj_and_critiera['Combined_Rk'] = (tes_best_proj_and_critiera['Rk_criteria'] * 0.3) - ((tes_best_proj_and_critiera['Projections']))
tes_best_proj_and_critiera = tes_best_proj_and_critiera.sort_values('Combined_Rk')
#wrs_best_proj_and_critiera_with_jjettas = wrs_best_proj_and_critiera[wrs_best_proj_and_critiera['Rk_criteria'] <= 33]
#print(wrs_best_proj_and_critiera_with_jjettas['Player'].unique()[:50])
#print(tes_best_proj_and_critiera['Player'].unique()[:20])
#tes_best_proj_and_critiera
tes_best_proj_and_critiera

,Player,Age_in_2024,Receiving_Ctch%,Receiving_Lng,Receiving_Y/Tgt,Shuttle,40yd,Ht,Wt,Vertical,3Cone,Rk_criteria,Rk_proj,Projections,Combined_Rk
0,Trey McBride,25,0.764,38,7.8,0.00,0.00,6.333333,246.0,33.0,0.00,5,22,12.248363,-10.748363
1,Dalton Schultz,28,0.670,31,7.2,4.40,4.75,6.416667,244.0,32.0,7.00,13,26,12.142769,-8.242769
3,Noah Fant,27,0.744,51,9.6,4.22,4.50,6.333333,249.0,39.5,6.81,28,4,14.742063,-6.342063
2,Luke Musgrave,24,0.739,37,7.7,0.00,0.00,6.500000,253.0,36.0,0.00,27,19,12.305000,-4.205000
4,Tucker Kraft,24,0.775,43,8.9,4.29,0.00,6.416667,254.0,34.0,7.08,30,11,13.063834,-4.063834
5,Michael Mayer,23,0.675,32,7.6,0.00,0.00,6.416667,249.0,32.5,0.00,36,35,11.912489,-1.112489


### TE Preferences
Short List:
- George Kittle
- Mark Andrews
- Sam LaPorta

Next Tier:
- Trey McBride
- Noah Fant
- Dalton Schultz
- Luke Musgrave

Flyers:
- Isaiah Likely
- Jonnu Smith
- Tucker Kraft
- Michael Mayer
- Cole Kmet

## Comparing ADP to List of Preferred Players above

In [70]:
preferred_players = pd.read_csv('draft_guides/2024/2024_preferred_players.csv')
adp = pd.read_csv('data/fantasy_draft_adp/draft_adp_2024.csv')


players_with_adp = preferred_players.merge(adp[['Name', '#','ADP','Overall', 'Std_Dev','High','Low']], on='Name', how='left')
#players_with_adp = adp.merge(preferred_players['Player', 'Position', 'Tier'], left='Name', right_on = 'Player')
players_with_adp['Round_10'] = np.ceil(players_with_adp['Overall'] / 10)
players_with_adp['Round_12'] = np.ceil(players_with_adp['Overall'] / 12)

In [79]:
# Sorted for list of average Overall ADP
players_with_adp_sorted_overall = players_with_adp.sort_values('Overall')
players_with_adp_sorted_overall.to_csv('draft_guides/2024/players_with_adp_sorted_overall.csv')

# Sorted by personal list of Tiers
players_with_adp_sorted_personal = players_with_adp.sort_values(['Tier', 'Round_12', 'High'])
players_with_adp_sorted_personal.to_csv('draft_guides/2024/players_with_adp_sorted_personal.csv')

## Probably worth placing some sort of score or projection on this to better grasp which players should be picked over another
## Would also be nice to see a chart of where different positions fall into which rounds so i know where its best to draft certain positions

In [78]:
players_with_adp_sorted_personal.head(30)

,Name,Position,Tier,#,ADP,Overall,Std_Dev,High,Low,Round_10,Round_12
21,CeeDee Lamb,WR,1,2.0,1.02,2.2,0.8,1.0,4.0,1.0,1.0
23,Tyreek Hill,WR,1,3.0,1.03,2.9,0.6,1.0,5.0,1.0,1.0
11,Breece Hall,RB,1,4.0,1.04,4.2,1.0,2.0,8.0,1.0,1.0
27,Ja'Marr Chase,WR,1,6.0,1.06,5.8,1.3,2.0,10.0,1.0,1.0
28,Amon-Ra St. Brown,WR,1,5.0,1.05,5.2,1.2,2.0,8.0,1.0,1.0
22,Justin Jefferson,WR,1,7.0,1.07,6.9,1.6,3.0,10.0,1.0,1.0
25,Puka Nacua,WR,1,13.0,1.13,12.8,2.2,7.0,17.0,2.0,2.0
13,Jonathan Taylor,RB,1,14.0,1.14,13.7,2.0,9.0,20.0,2.0,2.0
0,Josh Allen,QB,1,15.0,2.02,16.2,2.3,10.0,24.0,2.0,2.0
12,Isiah Pacheco,RB,1,17.0,2.04,18.0,2.2,12.0,23.0,2.0,2.0
